# Validação de sistemas de RAG

## 1) Carregamento de bibliotecas

In [10]:
import warnings

warnings.filterwarnings('ignore')

from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_classic.evaluation.qa import QAEvalChain
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer
from langchain_openai.embeddings import OpenAIEmbeddings
from operator import itemgetter
from dotenv import load_dotenv
import httpx
import os

http_client = httpx.Client(verify=False)

In [2]:
os.chdir(r'c:\Users\francisco.bneto\Documents\gen-ai-formation')
print(os.getcwd())

c:\Users\francisco.bneto\Documents\gen-ai-formation


In [7]:
_ = load_dotenv()
api_key = os.getenv('OPENROUTER_API_KEY')
api_url = os.getenv('OPENROUTER_BASE_URL')
print(f"Variáveis de ambiente carregadas: {os.getenv('OPENROUTER_API_KEY')[:10]}...")

Variáveis de ambiente carregadas: sk-or-v1-c...


In [11]:
# definição de modelos
embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=api_key,
    openai_api_base=api_url,
    http_client=http_client
)

llm_model = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    api_key=api_key,
    base_url=api_url,
    http_client=http_client
)

In [6]:
def avaliar(perguntas_respostas, geracoes):
    # perguntas_respostas: query, answer
    # geracoes: result
    avaliacoes = eval_chain(perguntas_respostas, geracoes)
    corretas = 0
    for i in enumerate(perguntas_respostas):
        corretas += (1 if avaliacoes[i]["results"].split("\n")[-1].split(":")[-1].strip() == "CORRECT" else 0)

    return corretas / len(perguntas_respostas)

## 2) Criação de chunks

In [12]:
# Carregamento de dados
print("Carregamento dos dados...")
pdfs = DirectoryLoader("./documentos", glob="*.pdf").load()

# Criação de chunks
print("Criação dos chunks...")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=1250,
    chunk_overlap=150
)

chunks = splitter.split_documents(pdfs)

# Criação do banco vetorial
print("Criação do banco vetorial...")
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

Carregamento dos dados...
Criação dos chunks...
Criação do banco vetorial...


## 3) Construção da chain de avaliação

In [ ]:
eval_chain = QAEvalChain.from_llm(llm_model)